In [1]:
print(2+2)

4


In [2]:
import json

# Définissez ici les chemins vers vos fichiers
fichier_entree = "C:\\users\\hp\\documents\\ia\\right-llm\\data\\train-data\\ohada_purifie.jsonl" # Remplacez par le nom de votre fichier
fichier_sortie = "donnees_nettoyees.jsonl"

def nettoyer_texte(texte):
    if not isinstance(texte, str):
        return texte
    
    # 1. Remplacer les apostrophes courbes/typographiques par la droite standard
    texte = texte.replace("’", "'")
    
    # 2. Supprimer les "---"
    texte = texte.replace("---", "")
    
    # 3. (Optionnel) Enlever les espaces inutiles laissés à la fin après suppression
    return texte.strip()

# Compteur pour vous donner un suivi
lignes_traitees = 0

with open(fichier_entree, 'r', encoding='utf-8') as f_in, \
     open(fichier_sortie, 'w', encoding='utf-8') as f_out:
    
    for ligne in f_in:
        # Ignorer les lignes vides
        if not ligne.strip():
            continue
            
        # Charger la ligne JSON
        donnees = json.loads(ligne)
        
        # Parcourir les messages et nettoyer le contenu
        if "messages" in donnees:
            for message in donnees["messages"]:
                if "content" in message:
                    message["content"] = nettoyer_texte(message["content"])
        
        # Réécrire la ligne nettoyée dans le nouveau fichier
        json.dump(donnees, f_out, ensure_ascii=False)
        f_out.write('\n')
        
        lignes_traitees += 1

print(f"✅ Terminé ! {lignes_traitees} lignes ont été nettoyées et sauvegardées dans '{fichier_sortie}'.")

✅ Terminé ! 259 lignes ont été nettoyées et sauvegardées dans 'donnees_nettoyees.jsonl'.


In [2]:
import json
import os

def supprimer_phrase_specifique(fichier_entree, fichier_sortie):
    # La phrase exacte que l'on veut cibler et supprimer
    phrase_a_bannir = "Que dit cet article en droit OHADA ?"
    
    if not os.path.exists(fichier_entree):
        print(f"❌ Erreur : Le fichier {fichier_entree} n'existe pas.")
        return

    stats = {
        "totales": 0,
        "conservees": 0,
        "supprimees": 0,
        "erreurs": 0
    }

    print(f"🛠️ Démarrage du filtrage sur : {fichier_entree}")
    
    with open(fichier_entree, 'r', encoding='utf-8') as infile, \
         open(fichier_sortie, 'w', encoding='utf-8') as outfile:
        
        for ligne in infile:
            stats["totales"] += 1
            ligne = ligne.strip()
            
            if not ligne:
                continue
                
            try:
                data = json.loads(ligne)
                doit_etre_supprimee = False
                
                # On parcourt les messages pour trouver celui de l'utilisateur
                if "messages" in data and isinstance(data["messages"], list):
                    for msg in data["messages"]:
                        # Si le rôle est 'user' et que le texte correspond exactement à la phrase à bannir
                        if msg.get("role") == "user" and msg.get("content", "").strip() == phrase_a_bannir:
                            doit_etre_supprimee = True
                            break # On a trouvé la phrase, on arrête de chercher dans cette ligne
                
                # Si on a détecté la phrase, on ne l'écrit pas (donc elle est supprimée)
                if doit_etre_supprimee:
                    stats["supprimees"] += 1
                else:
                    # Sinon, la ligne est saine, on la sauvegarde
                    json.dump(data, outfile, ensure_ascii=False)
                    outfile.write('\n')
                    stats["conservees"] += 1
                    
            except json.JSONDecodeError:
                stats["erreurs"] += 1
                
    # --- RAPPORT FINAL ---
    print("\n✅ Nettoyage terminé avec succès !")
    print("📊 --- RAPPORT ---")
    print(f"🔹 Lignes analysées : {stats['totales']}")
    print(f"💾 Lignes CONSERVÉES : {stats['conservees']}")
    print(f"🗑️ Lignes SUPPRIMÉES : {stats['supprimees']} (Contenaient la phrase exacte)")
    if stats['erreurs'] > 0:
        print(f"⚠️ Erreurs de lecture JSON : {stats['erreurs']}")
    print(f"📁 Fichier nettoyé sauvegardé sous : {fichier_sortie}")


# ==========================================
# EXÉCUTION DU SCRIPT
# ==========================================
# Remplacez par les noms de vos fichiers
FICHIER_SOURCE = "donnees_nettoyees.jsonl" 
FICHIER_PROPRE = "donnees_nettoyees.jsonl" 

supprimer_phrase_specifique(FICHIER_SOURCE, FICHIER_PROPRE)

🛠️ Démarrage du filtrage sur : donnees_nettoyees.jsonl

✅ Nettoyage terminé avec succès !
📊 --- RAPPORT ---
🔹 Lignes analysées : 0
💾 Lignes CONSERVÉES : 0
🗑️ Lignes SUPPRIMÉES : 0 (Contenaient la phrase exacte)
📁 Fichier nettoyé sauvegardé sous : donnees_nettoyees.jsonl
